# 🌿 Python Trie — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A Trie is like a building with floors. The lobby is empty. Each floor branches into up to 26 doors,
> one per letter. To store the word "cat", you go through door 'c', then door 'a', then door 't',
> and leave a flag on that room saying "a word ends here."
> To look up a prefix, you just walk the floors — if every door exists, the prefix is there.
> Words that share a prefix share the same hallway.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is a Trie? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Implement Trie (LC 208)](#5) |
| 6 | [Pattern 2: Add and Search Words / Wildcard (LC 211)](#6) |
| 7 | [Pattern 3: Word Search II / Trie + DFS (LC 212)](#7) |
| 8 | [Pattern 4: Maximum XOR / Bit Trie (LC 421)](#8) |
| 9 | [The Trie Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is a Trie? The Visual Model

```
               TRIE — THE HALLWAY BUILDING

  Words inserted: "cat", "car", "card", "care", "bat"

  root
  ├── c
  │   └── a
  │       ├── t [END]     ← "cat" ends here
  │       └── r [END]     ← "car" ends here
  │           ├── d [END] ← "card" ends here
  │           └── e [END] ← "care" ends here
  └── b
      └── a
          └── t [END]     ← "bat" ends here

  OPERATIONS:
  insert("car"):  root→c→a→r, mark r as END
  search("car"):  root→c→a→r, r is END → True
  search("ca"):   root→c→a,   a is NOT END → False
  startsWith("ca"): root→c→a, path exists → True

  IMPLEMENTATION CHOICES:
  Option A: TrieNode class with children dict + is_end flag
  Option B: defaultdict of dicts (nested dicts)
  Option C: array of 26 (fixed lowercase alphabet)

  COMPLEXITY:
  insert / search / startsWith: O(L) — L = word length
  Space: O(ALPHABET_SIZE * L * N) — N words, L avg length
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# TrieNode — the building block for every Trie
class TrieNode:
    def __init__(self):
        self.children = {}   # char → TrieNode (use dict for sparse alphabets)
        self.is_end   = False  # True if a word terminates at this node

# Full Trie class — Option A (TrieNode objects)
class Trie:
    def __init__(self):
        self.root = TrieNode()   # empty lobby — no letters yet

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()  # open a new door
            node = node.children[ch]             # walk through the door
        node.is_end = True                       # plant the END flag

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end  # path exists AND word ends here

    def starts_with(self, prefix):
        return self._walk(prefix) is not None    # path exists → prefix found

    def _walk(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return None              # door doesn't exist → word/prefix absent
            node = node.children[ch]
        return node                      # return the node at the end of the path

# Option B — nested defaultdict Trie (compact, no class needed)
from collections import defaultdict

def make_trie():
    return defaultdict(make_trie)  # each node is a defaultdict of more nodes

# Demo
trie = Trie()
for word in ["cat", "car", "card", "care", "bat"]:
    trie.insert(word)
print("search('car'):",     trie.search("car"))      # True
print("search('ca'):",      trie.search("ca"))       # False (no END)
print("startsWith('ca'):",  trie.starts_with("ca"))  # True
print("startsWith('xyz'):", trie.starts_with("xyz")) # False
print("Trie class loaded.")

<a id='3'></a>
## 3. The Core API — All Operations

```
OPERATION          COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────────
insert(word)       O(L)         walk/create nodes char by char; mark end
search(word)       O(L)         walk path; True only if path exists AND is_end
starts_with(pfx)   O(L)         walk path; True if path exists (no end check)
delete(word)       O(L)         mark end=False; optionally prune leaf nodes
_walk(word)        O(L)         shared traversal helper; returns final node or None

THINGS YOU DO NOT DO:
❌  Return True for search() when path exists but is_end is False
     ("ca" is a PREFIX of "cat" — it's NOT a valid word unless explicitly inserted)
❌  Use a list of 26 for children when alphabet is unknown or large (use dict)
❌  Forget to handle wildcards ('.' in search) as a recursive branch over all children
❌  Modify the trie while DFS-ing through it (Word Search II — use a ref count trick)
```

In [ ]:
# Live demo of all core operations
demo_trie = Trie()

# INSERT words one by one
for w in ["apple", "app", "apex", "banana"]:
    demo_trie.insert(w)
    print(f"inserted '{w}'")

# SEARCH — must be exact word with is_end = True
print("search('app'):",    demo_trie.search("app"))    # True  — inserted as own word
print("search('ap'):",     demo_trie.search("ap"))     # False — only a prefix
print("search('apple'):",  demo_trie.search("apple"))  # True
print("search('apples'):", demo_trie.search("apples")) # False — not inserted

# STARTS_WITH — prefix check, no end requirement
print("startsWith('ap'):", demo_trie.starts_with("ap"))     # True
print("startsWith('ban'):", demo_trie.starts_with("ban"))   # True
print("startsWith('xyz'):", demo_trie.starts_with("xyz"))   # False

print("Core API demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"implement Trie / prefix tree"          TrieNode + insert/search/startsWith
"word search with wildcards '.'"        Trie + DFS branching on all children
"find all words on a board"             Trie + grid DFS (Word Search II)
"longest common prefix"                 Insert all words, walk until branch
"autocomplete / type-ahead"             Trie + DFS/BFS to collect words
"maximum XOR of pairs"                  Bit Trie (32 bits as path, max bit greedily)
"word break / can form word from dict" Trie + DP
"count distinct prefixes"               Trie — count nodes on insert paths
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Implement Trie — LC 208

---

```
PROBLEM:
  Implement a Trie data structure with insert(word), search(word),
  and startsWith(prefix) methods.

TRICK:
  TrieNode holds a children dict (char→node) and an is_end flag.
  All three operations share the same walk-path logic.
  search needs is_end=True at the final node; startsWith only needs the path to exist.

SLOW MOTION TRACE — insert "app", then search:

  insert("app"):
    root → create 'a' node
    'a'  → create 'p' node
    'p'  → create 'p' node
    mark last 'p' as is_end=True

  search("app"):    walk root→a→p→p, is_end=True  → True
  search("ap"):     walk root→a→p,   is_end=False → False
  startsWith("ap"): walk root→a→p,   path exists  → True
  search("appl"):   walk root→a→p→p, no 'l' child → False

KEY INSIGHT:
  is_end is the dividing line between "word" and "prefix".
  A prefix that was never inserted as a word has is_end=False.

TIME:  O(L) per operation — L = length of word/prefix
SPACE: O(SIGMA * N * L) — SIGMA=alphabet size, N=word count, L=avg length
```

In [ ]:
class TrieLC208:
    """
    LC 208 — Implement Trie (Prefix Tree)
    Approach: TrieNode with children dict and is_end flag; shared _walk helper.
    Time:  O(L) per insert/search/startsWith  — L = word/prefix length
    Space: O(SIGMA * N * L) — all inserted characters stored as nodes
    """
    def __init__(self):
        self.root = TrieNode()   # empty root — no characters yet

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()  # new hallway door for this char
            node = node.children[ch]             # step through the door
        node.is_end = True                       # this room = word's final stop

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end  # path AND word-ending flag

    def startsWith(self, prefix):
        return self._walk(prefix) is not None    # path existence is enough

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None              # door missing → prefix/word absent
            node = node.children[ch]
        return node

# Slow motion on ["app", search "app", search "ap", startsWith "ap"]:
# insert app: root→{a:Node}→{p:Node}→{p:Node is_end=True}
# search app: walk a→p→p, is_end=True → True
# search ap:  walk a→p, is_end=False → False
# startsWith ap: walk a→p, not None → True

def test_harness(TrieClass):
    ops = [
        ("insert",     "apple",  None),
        ("search",     "apple",  True),
        ("search",     "app",    False),   # not inserted as separate word
        ("startsWith", "app",    True),
        ("insert",     "app",    None),
        ("search",     "app",    True),    # now inserted
        ("search",     "xyz",    False),
        ("startsWith", "xyz",    False),
    ]
    t = TrieClass()
    passed = 0
    total = 0
    for op, word, expected in ops:
        if op == "insert":
            t.insert(word)
        elif op == "search":
            got = t.search(word)
            total += 1
            status = "PASSED" if got == expected else "FAILED"
            if status == "FAILED":
                print(f"{status} | search('{word}') expected={expected} got={got}")
            passed += (got == expected)
        else:
            got = t.startsWith(word)
            total += 1
            status = "PASSED" if got == expected else "FAILED"
            if status == "FAILED":
                print(f"{status} | startsWith('{word}') expected={expected} got={got}")
            passed += (got == expected)
    print(f"{passed}/{total} tests passed")

test_harness(TrieLC208)
print("TrieLC208 defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Add and Search Words / Wildcard — LC 211

---

```
PROBLEM:
  Design a data structure that supports addWord(word) and search(word),
  where word can contain '.' as a wildcard matching any single character.

TRICK:
  Same Trie structure as LC 208. The only change: in search(),
  when you encounter '.', branch recursively into ALL children
  (any of them might match). Use DFS for the branching.

SLOW MOTION TRACE — addWord "bad", search ".ad":

  Trie after addWord "bad":
  root → {b: Node} → {a: Node} → {d: Node is_end=True}

  search(".ad"):
    pos=0, ch='.': try ALL children of root → {b}
      enter node 'b'
    pos=1, ch='a': 'a' in children? YES → enter 'a'
    pos=2, ch='d': 'd' in children? YES → enter 'd'
    pos=3: end of word, is_end=True → True

  search("b.."):
    b→ enter, '.' → try all children {a} → a→ enter, '.' → try all {d} → d is_end → True

KEY INSIGHT:
  '.' = branch on ALL children at that level. Continue DFS on each branch.
  Return True as soon as any branch succeeds.

TIME:  O(L) best (no wildcards), O(SIGMA^L) worst (all wildcards)
SPACE: O(L * N) — trie storage, O(L) recursion stack
```

In [ ]:
class WordDictionary:
    """
    LC 211 — Design Add and Search Words Data Structure
    Approach: Standard Trie; wildcard '.' triggers DFS over all children.
    Time:  addWord O(L); search O(L) no wildcards, O(SIGMA^L) all wildcards
    Space: O(N*L) — trie nodes; O(L) recursion stack
    """
    def __init__(self):
        self.root = TrieNode()

    def addWord(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        def dfs(node, pos):
            if pos == len(word):
                return node.is_end      # consumed all chars → check if word ends here
            ch = word[pos]
            if ch == '.':
                # wildcard — try every child at this level
                return any(dfs(child, pos + 1) for child in node.children.values())
            if ch not in node.children:
                return False            # no door for this char — dead end
            return dfs(node.children[ch], pos + 1)  # step through the door
        return dfs(self.root, 0)

# Slow motion on addWord ["bad","dad","mad"], search '.ad':
# dfs(root, 0): ch='.', try all children {b,d,m}
#   dfs(b_node, 1): ch='a', go to a_node
#     dfs(a_node, 2): ch='d', go to d_node
#       dfs(d_node, 3): pos==len → is_end=True → True (short-circuit)
# → True

def test_harness(WDClass):
    wd = WDClass()
    for w in ["bad", "dad", "mad"]:
        wd.addWord(w)
    tests = [
        ("pad", False),
        ("bad", True),
        (".ad", True),
        ("b..", True),
        ("...", True),
        ("....", False),  # length 4, no words of length 4
        ("b.d", True),
        ("p..", False),
    ]
    passed = 0
    for word, expected in tests:
        got = wd.search(word)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | search('{word}') expected={expected} got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(WordDictionary)
print("WordDictionary defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Word Search II — Trie + Grid DFS — LC 212

---

```
PROBLEM:
  Given a board of characters and a list of words, find all words on the board.
  Words are formed by sequentially adjacent cells (4-directional, no reuse).

TRICK:
  Insert all words into a Trie. Then DFS the board using the Trie as a guide:
  - At each cell, follow the Trie path matching board letters.
  - If current Trie node has is_end=True → record the word.
  - If current board char is not in trie node's children → prune immediately.
  This avoids checking each word separately — O(W*L) vs Trie-guided DFS.

SLOW MOTION TRACE on words=["oath","pea"] and small board:

  Trie:
    root → o → a → t → h [END=oath]
         → p → e → a [END=pea]

  DFS from each cell:
  cell (0,0)='o': 'o' in root.children → enter trie node for 'o'
    explore neighbors for 'a' → found path o→a→t→h → record "oath"
  cell for 'p' → enter trie node for 'p' → find p→e→a → record "pea"

  After recording a word: mark is_end=False (pruning — avoid duplicates)

KEY INSIGHT:
  Trie prunes the DFS: if a prefix doesn't exist in the Trie, abandon that branch.
  Without Trie: O(N*4^L) per word. With Trie: shared prefix exploration.

TIME:  O(M * N * 4^L) — M*N board cells, L = max word length (with pruning much less)
SPACE: O(W*L) — trie storage for W words of avg length L
```

In [ ]:
DIRS = [(0,1),(0,-1),(1,0),(-1,0)]

def find_words(board, words):
    """
    LC 212 — Word Search II
    Approach: Build Trie from word list; DFS board using Trie to prune dead branches.
    Args:
        board (List[List[str]]): m x n grid of characters.
        words (List[str]): words to find on the board.
    Returns:
        List[str]: all words from the list found on the board.
    Time:  O(M*N*4^L) — M*N cells, L=max word length; Trie pruning reduces significantly
    Space: O(W*L) — Trie for W words; O(L) recursion stack
    """
    # build trie from all target words
    root = TrieNode()
    for word in words:
        node = root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True
        node.word = word             # store the full word at the end node

    rows, cols = len(board), len(board[0])
    found = []

    def dfs(r, c, trie_node):
        ch = board[r][c]
        if ch not in trie_node.children:
            return                   # trie doesn't recognize this letter — prune
        next_node = trie_node.children[ch]
        if next_node.is_end:
            found.append(next_node.word)   # full word found on the board!
            next_node.is_end = False       # mark used to avoid re-adding duplicates

        board[r][c] = '#'            # mark cell as in-use (no revisiting this path)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and board[nr][nc] != '#':
                dfs(nr, nc, next_node)   # explore adjacent cell using next trie level
        board[r][c] = ch             # restore cell for other DFS paths (backtrack)

    for r in range(rows):
        for c in range(cols):
            dfs(r, c, root)          # start DFS from every cell with root as guide
    return found

# Slow motion on board and words=["oath","pea"]:
# insert into trie: o→a→t→h[END,word=oath], p→e→a[END,word=pea]
# dfs from each cell; when trie node says END → record; mark '#' to prevent revisit
# restore after each DFS branch (backtracking)

def test_harness(fn):
    tests = [
        (
            [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]],
            ["oath","pea","eat","rain"],
            sorted(["eat","oath"])
        ),
        (
            [["a","b"],["c","d"]],
            ["abdc"],
            ["abdc"]
        ),
        (
            [["a"]],
            ["a"],
            ["a"]
        ),
        (
            [["a","b"],["c","d"]],
            ["xyz"],
            []
        ),
    ]
    passed = 0
    import copy
    for board, words, expected in tests:
        board_copy = copy.deepcopy(board)
        got = sorted(fn(board_copy, words))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | words={words} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_words)
print("find_words defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Maximum XOR of Two Numbers — Bit Trie — LC 421

---

```
PROBLEM:
  Given an integer array nums, find the maximum XOR of any two elements.

TRICK:
  Build a BINARY TRIE where each path from root to leaf encodes a 32-bit number
  (most significant bit first). For each number, greedily walk the trie
  choosing the OPPOSITE bit at each level (to maximize XOR = maximize differences).
  If the opposite bit branch doesn't exist, take the same bit (XOR gives 0 there).

SLOW MOTION TRACE on nums=[3,10,5,25,2,8]:

  Binary representations (4 bits for simplicity):
  3  = 0011
  10 = 1010
  5  = 0101
  25 = 11001 (5 bits)

  For number=5 (0101), walk the trie greedily:
  bit 4: 5's bit=0, want 1 (opposite) → check if 1-branch exists → 10,25 have it → go 1
  bit 3: 5's bit=1, want 0 → check 0-branch → 10=1010 has 0 here → go 0
  bit 2: 5's bit=0, want 1 → 10=1010 has 1 here → go 1
  bit 1: 5's bit=1, want 0 → 10=1010 has 0 here → go 0
  XOR with 10 = 5 XOR 10 = 15

  Best pair is 5 XOR 25 = 28.

KEY INSIGHT:
  XOR is maximized by maximizing the most significant differing bit.
  Bit Trie lets us make this greedy choice in O(32) per number.

TIME:  O(N * 32) = O(N) — N numbers, 32-bit depth
SPACE: O(N * 32) — trie nodes
```

In [ ]:
class BitTrieNode:
    def __init__(self):
        self.children = {}   # keys are 0 or 1 (bits)

def find_maximum_xor(nums):
    """
    LC 421 — Maximum XOR of Two Numbers in an Array
    Approach: Binary Trie; for each num greedily pick opposite bits to maximize XOR.
    Args:
        nums (List[int]): array of non-negative integers.
    Returns:
        int: maximum XOR value achievable from any pair.
    Time:  O(N * 32) = O(N) — insert and query each of N numbers, 32 bits deep
    Space: O(N * 32)        — trie holds at most N*32 nodes
    """
    root = BitTrieNode()
    MAX_BITS = 31    # treat all numbers as 32-bit (bit 31 down to bit 0)

    def insert(num):
        node = root
        for i in range(MAX_BITS, -1, -1):   # MSB first
            bit = (num >> i) & 1             # extract bit i
            if bit not in node.children:
                node.children[bit] = BitTrieNode()
            node = node.children[bit]

    def query_max_xor(num):
        node = root
        xor_val = 0
        for i in range(MAX_BITS, -1, -1):
            bit = (num >> i) & 1             # current bit of num
            want = 1 - bit                   # opposite bit → maximizes XOR at this position
            if want in node.children:
                xor_val |= (1 << i)          # this bit of XOR is 1 (different bits)
                node = node.children[want]
            else:
                node = node.children[bit]    # forced to take same bit → XOR bit = 0
        return xor_val

    # build the trie from all numbers
    for num in nums:
        insert(num)

    # for each number, find its best XOR partner in the trie
    max_xor = 0
    for num in nums:
        max_xor = max(max_xor, query_max_xor(num))
    return max_xor

# Slow motion on [3, 10, 5, 25, 2, 8]:
# 5 XOR 25 = 0b00101 XOR 0b11001 = 0b11100 = 28
# 2 XOR 25 = 0b00010 XOR 0b11001 = 0b11011 = 27
# max XOR = 28 (pair 5, 25)

def test_harness(fn):
    tests = [
        ([3,10,5,25,2,8], 28),     # 5 XOR 25
        ([14,70,53,83,49,91,36,80,92,51,66,70], 127),
        ([0], 0),                   # single element
        ([0,0], 0),
        ([1,2], 3),                 # 1 XOR 2 = 3
        ([4,6,7], 3),               # 4 XOR 7 = 3
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | nums={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_maximum_xor)
print("find_maximum_xor defined.")

<a id='9'></a>
## 9. The Trie Decision Map

```
QUESTION TYPE                         KEY TECHNIQUE                 LC PROBLEMS
───────────────────────────────────────────────────────────────────────────────
Prefix search / startsWith            Standard Trie                  208
Wildcard search (dot = any char)      Trie + DFS branching on '.'    211
Find all words on grid                Trie + grid DFS (prune early)  212
Maximum XOR of pairs                  Bit Trie (32-level binary)     421
Longest common prefix                 Insert all → walk until fork   14
Autocomplete / word suggestion        Trie → DFS collect all words   —
Word break (can split string)         Trie + DP                      139
Count words with given prefix         Trie + count field at nodes    —

WHEN TRIE > HASH SET:
  Prefix queries:   Trie is O(L); hash set has no prefix support
  Wildcard search:  Trie branches at '.'; hash set can't
  Board search:     Trie prunes DFS paths early using the tree structure
  Bit-level ops:    Bit Trie exploits greedy bit-by-bit decisions
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for Trie:**

| Signal | What to Do |
|--------|------------|
| "prefix search" or "startsWith" | Standard Trie + is_end flag |
| wildcard '.' matching | Trie search with DFS branching |
| "find words on a board" | Trie + grid DFS backtracking |
| "maximum XOR" | 32-level bit Trie, greedy opposite bit |
| "autocomplete" | Trie + collect all words via DFS |

**2. The O(L) operations — memorize these:**

```python
# TRIE NODE
class TrieNode:
    def __init__(self):
        self.children = {}    # char → TrieNode
        self.is_end   = False

# INSERT
def insert(root, word):
    node = root
    for ch in word:
        if ch not in node.children:
            node.children[ch] = TrieNode()
        node = node.children[ch]
    node.is_end = True

# SEARCH (exact)
def search(root, word):
    node = root
    for ch in word:
        if ch not in node.children: return False
        node = node.children[ch]
    return node.is_end

# STARTS WITH
def starts_with(root, prefix):
    node = root
    for ch in prefix:
        if ch not in node.children: return False
        node = node.children[ch]
    return True
```

**3. Common templates:**

```python
# TEMPLATE: WILDCARD SEARCH (LC 211)
def search_wild(node, word, pos):
    if pos == len(word): return node.is_end
    if word[pos] == '.':
        return any(search_wild(c, word, pos+1) for c in node.children.values())
    if word[pos] not in node.children: return False
    return search_wild(node.children[word[pos]], word, pos+1)

# TEMPLATE: WORD SEARCH II (LC 212) — DFS + Trie pruning
def dfs(board, r, c, trie_node, found):
    ch = board[r][c]
    if ch not in trie_node.children: return
    nxt = trie_node.children[ch]
    if nxt.is_end:
        found.append(nxt.word); nxt.is_end = False   # avoid duplicates
    board[r][c] = '#'            # mark visited
    for dr, dc in DIRS:
        nr, nc = r+dr, c+dc
        if 0 <= nr < R and 0 <= nc < C and board[nr][nc] != '#':
            dfs(board, nr, nc, nxt, found)
    board[r][c] = ch             # restore (backtrack)

# TEMPLATE: BIT TRIE (LC 421)
for bit in range(31, -1, -1):
    b = (num >> bit) & 1
    want = 1 - b
    if want in node.children: xor_val |= (1<<bit); node = node.children[want]
    else: node = node.children[b]
```

**4. Gotchas to not forget:**

```
❌  Returning True in search() when is_end is False (prefix ≠ word)
❌  Forgetting to store the full word at the END node in Word Search II
❌  Not backtracking board[r][c] after DFS in Word Search II
❌  In Bit Trie: going MSB→LSB wrong direction (always MSB first for greedy XOR)
✅  Mark is_end=False after finding a word in Word Search II (prevents duplicates)
✅  Use dict children for sparse alphabets; array[26] only for lowercase a-z
✅  _walk() helper avoids code duplication between search() and startsWith()
✅  Bit Trie depth = 32 for 32-bit integers; adjust for problem constraints
```

## Summary Map

```
                    🌿 TRIE (PREFIX TREE)
                            │
           ┌────────────────┼────────────────┐
           │                │                │
       CHAR TRIE        BIT TRIE         GRID TRIE
           │             32 levels        Trie + DFS
      ┌────┴────┐            │           backtracking
      │         │        MAX XOR          LC 212
   EXACT     PREFIX      greedy
   SEARCH    SEARCH      opposite
   is_end    no end      bit
   LC 208    check       LC 421
               │
           WILDCARD
           '.' → DFS
           all children
           LC 211

CORE RULE:
  Each node = one character on a path. is_end = a complete word terminates here.
  Prefix traversal = walk until path ends or char missing.
  Trie trades space for O(L) time — faster than O(L*N) naive search over N words.
```

---
*End of Trie Master Guide — Sean Edition*